# Visualize results

This is a notebook to visualize the results of different metric calculations for different datasets and conditions, for AMPLIFY model checkpoints. We want to see:

**Questions:**
- For each model, how do the scores change as we go further into the model (increasing depth)?
- Do different models have different patterns in how their scores change through the layers, for a given dataset?


In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import warnings

# This will ignore all UserWarning messages
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import re
from functools import reduce

from project.utils.strs import linear_probe_metrics_dir, linear_probe_figures_dir, linear_probe_compiled_results_dir

In [ ]:
# Make dirs if they don't exist already
for directory in [linear_probe_metrics_dir, linear_probe_figures_dir, linear_probe_compiled_results_dir]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
datasets = [
    'prot_param',
    'uniprot_peptide',
    'interpro_conserved_site',
    'biomap_localization_prediction',
    'interpro_repeat',
    'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    'interpro_homologous_superfamily',
    'uniprot_functional_sites',
    'interpro_binding_site',
    'biomap_metal_ion_binding',
    'interpro_active_site',
    'uniprot_topology',
    'uniprot_post_translational_modification',
    'uniprot_phosphorylation',
    'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

protein_feature_colormapping = {
    # Baseline 1D (Grey)
    "prot_param": "#708090",                       # SlateGray (Baseline)

    # 1D: Sequence Signals (Orange Flow)
    "uniprot_peptide": "#FFB88C",                   # Light Orange
    "interpro_conserved_site": "#FF8C00",           # DarkOrange
    "biomap_localization_prediction": "#E65100",    # Deep Burnt Orange

    # 2D: Secondary Structure (Blue Flow)
    "interpro_repeat": "#A2D2FF",                   # Light Sky Blue
    "uniprot_secondary_structure": "#5D9CEC",       # Soft Blue
    "biomap_ssp_q3": "#3498DB",                     # Bright Blue
    "biomap_ssp_q8": "#1E3A8A",                     # Deep Royal Blue (Flowing toward 3D)
    
    # 3D: Pure Structural Architecture (Light Green Flow)
    "interpro_homologous_superfamily": "#D1FAE5",  # Mint Cream
    "interpro_family": "#A7F3D0",                   # Pale Emerald
    "interpro_domain": "#6EE7B7",                   # Light Sea Green
    "uniprot_topology": "#34D399",                  # Medium Emerald

    # 3D + System Hybrid: Interactive/Active (Dark Green Flow)
    "uniprot_functional_sites": "#065F46",          # Dark Emerald
    "interpro_binding_site": "#064E3B",             # Deep Hunter Green
    "biomap_metal_ion_binding": "#022C22",          # Near-Black Green
    "interpro_active_site": "#0F172A",              # Deepest Green-Grey
    "uniprot_post_translational_modification": "#1E3A1A", # Dark Forest
    "uniprot_phosphorylation": "#14532D",           # Rich Moss Green
    "uniprot_lipidation": "#064E3B",                # Deep Evergreen

    # 4D+: System-Level Biological Process (Magenta Flow)
    "GO_cc": "#F0ABFC",                             # Light Orchid
    "GO_mf": "#D946EF",                             # Steel Magenta
    "GO_bp": "#701A75",                             # Deep Plum/Grape
}

model_rename_dict = {
    'amplify_120m': 'AMPLIFY 120M',
    'samplify_120m': 'SaAMPLIFY 120M',
    'amplify_350m': 'AMPLIFY 350M',
    'samplify_350m': 'SaAMPLIFY 350M',
    'esm2_8m': 'ESM2 8M',
    'esm2_35m': 'ESM2 35M',
    'esm2_150m': 'ESM2 150M',
    'esm2_650m': 'ESM2 650M',
    'mila_amplify_120m_100000': 'AMPLIFY 120M UR100 100k',
    'mila_amplify_120m_200000': 'AMPLIFY 120M UR100 200k',
    'mila_amplify_120m_300000': 'AMPLIFY 120M UR100 300k',
    'mila_amplify_120m_400000': 'AMPLIFY 120M UR100 400k',
    'mila_amplify_120m_500000': 'AMPLIFY 120M UR100 500k',
    'mila_amplify_120m_600000': 'AMPLIFY 120M UR100 600k',
    'mila_amplify_120m_700000': 'AMPLIFY 120M UR100 700k',
    'mila_amplify_120m_800000': 'AMPLIFY 120M UR100 800k',
    'mila_amplify_120m_900000': 'AMPLIFY 120M UR100 900k',
    'mila_amplify_120m_1000000': 'AMPLIFY 120M UR100 1M',
    'mila_amplify_120m_u50_100000': 'AMPLIFY 120M UR50 100k',
    'mila_amplify_120m_u50_200000': 'AMPLIFY 120M UR50 200k',      
    'mila_amplify_120m_u50_300000': 'AMPLIFY 120M UR50 300k',
    'mila_amplify_120m_u50_400000': 'AMPLIFY 120M UR50 400k',
    'mila_amplify_120m_u50_500000': 'AMPLIFY 120M UR50 500k',
    'mila_amplify_120m_u50_600000': 'AMPLIFY 120M UR50 600k',
    'mila_amplify_120m_u50_700000': 'AMPLIFY 120M UR50 700k',
    'mila_amplify_120m_u50_800000': 'AMPLIFY 120M UR50 800k',
    'mila_amplify_120m_u50_900000': 'AMPLIFY 120M UR50 900k',
    'mila_amplify_120m_u50_1000000': 'AMPLIFY 120M UR50 1M'
}

model_colormapping = {
    'amplify_120m': '#FF0000',
    'mila_amplify_120m_100000': '#D3D3D3',
    'mila_amplify_120m_200000': '#BEBEBE',
    'mila_amplify_120m_300000': '#A9A9A9',
    'mila_amplify_120m_400000': '#949494',
    'mila_amplify_120m_500000': '#808080',
    'mila_amplify_120m_600000': '#6B6B6B',
    'mila_amplify_120m_700000': '#565656',
    'mila_amplify_120m_800000': '#414141',
    'mila_amplify_120m_900000': '#2C2C2C',
    'mila_amplify_120m_1000000': '#171717',
    'mila_amplify_120m_u50_100000': '#D3D3D3',
    'mila_amplify_120m_u50_200000': '#BEBEBE',      
    'mila_amplify_120m_u50_300000': '#A9A9A9',
    'mila_amplify_120m_u50_400000': '#949494',
    'mila_amplify_120m_u50_500000': '#808080',
    'mila_amplify_120m_u50_600000': '#6B6B6B',
    'mila_amplify_120m_u50_700000': '#565656',
    'mila_amplify_120m_u50_800000': '#414141',
    'mila_amplify_120m_u50_900000': '#2C2C2C',
    'mila_amplify_120m_u50_1000000': '#171717'
}

dataset_rename_dict = {
    # Physical & Sequence Properties
    'prot_param': 'Protein Parameters (Physicochemical)',
    'uniprot_peptide': 'Peptide Signal Sequences',
    'biomap_localization_prediction': 'Subcellular Localization',
    
    # Secondary & Local Structure
    'biomap_ssp_q3': 'Secondary Structure (3-class)',
    'biomap_ssp_q8': 'Secondary Structure (8-class)',
    'uniprot_secondary_structure': 'Secondary Structure (UniProt)',
    'uniprot_topology': 'Transmembrane Topology',
    
    # Domains & Families
    'interpro_domain': 'InterPro Domains',
    'interpro_family': 'InterPro Families',
    'interpro_homologous_superfamily': 'Homologous Superfamilies',
    'interpro_repeat': 'Structural Repeats',
    
    # Sites & Modifications
    'interpro_conserved_site': 'Conserved Sites',
    'interpro_binding_site': 'Binding Sites',
    'interpro_active_site': 'Enzymatic Active Sites',
    'uniprot_functional_sites': 'Functional Sites',
    'biomap_metal_ion_binding': 'Metal Ion Binding',
    'uniprot_post_translational_modification': 'PTMs',
    'uniprot_phosphorylation': 'Phosphorylation Sites',
    'uniprot_lipidation': 'Lipidation Sites',
    
    # Gene Ontology
    'GO_cc': 'GO Cellular Component',
    'GO_mf': 'GO Molecular Function',
    'GO_bp': 'GO Biological Process',
}

Read in the results

In [ ]:
mean_dfs = []
for dataset in datasets:
    for d in linear_probe_metrics_dir.glob(f"*{dataset}*"):
        # Filter out older datasets we don't want
        if ('tiny' not in str(d)) and ('medium' not in str(d)) and ('512' in (str(d))):
            try:
                dataset_dfs = []
                for f in d.iterdir():
                    dataset_dfs.append(pl.read_parquet(f)) 
                # Combine
                df = pl.concat(dataset_dfs, how='diagonal_relaxed')
                # Only select columns that don't have elementwise metrics
                mean_dfs.append(df.select([c for c in df.columns if ('elementwise' not in c)]))

            except Exception as e:
                print(f"problem with {f}: {e}")

mean_df = pl.concat(mean_dfs, how='diagonal_relaxed')

# Adjust the dataset nomenclature and extract metadata
mean_df = mean_df.with_columns([
    # 1. Extract '512_cutoff' if present, otherwise default to 'standard' (or null)
    pl.col("dataset")
    .str.extract(r"_(512_cutoff)", 1)
    .fill_null("standard")
    .alias("data_subset"),

    # 2. Extract the split number
    pl.col("dataset")
    .str.extract(r"_split(\d+)$", 1)
    .fill_null("0")
    .alias("split"),

    # 3. Clean the dataset name by removing the split and the cutoff suffixes
    pl.col("dataset")
    .str.replace(r"_512_cutoff", "")  # Remove cutoff
    .str.replace(r"_split\d+$", "")   # Remove split
    .alias("dataset")
]).with_columns(
    pl.col("split").cast(pl.Int64)
)

In [ ]:
mean_df

In [ ]:
mean_df.sort(by=['dataset', 'model_name', 'layer_num', 'plm_state', 'plm_embeddings_normalized', 'linear_probe_state', 'control_type']).write_parquet(linear_probe_compiled_results_dir / 'compiled_checkpoints_mean_results_all_probes.parquet.gz')

### Summary figure: all datasets, all models, all layers, all controls

### Controls for a given model 

In [ ]:
# Harmonize 'score' column for plotting
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained', 'un-trained']
probe_states = ['trained', 'un-trained']
show_controls = ['original', 'scrambled', 'mean', 'l2_normalized', 'random_gaussian'] #
model_order = [
    # 'amplify_120m',
    'mila_amplify_120m_100000',
    'mila_amplify_120m_200000',
    'mila_amplify_120m_300000',
    'mila_amplify_120m_400000',
    'mila_amplify_120m_500000',
    'mila_amplify_120m_600000',
    'mila_amplify_120m_700000',
    'mila_amplify_120m_800000',
    'mila_amplify_120m_900000',
    'mila_amplify_120m_1000000'
    ]
dataset_order = [
    'prot_param',
    'uniprot_peptide',
    'interpro_conserved_site',
    'biomap_localization_prediction',
    'interpro_repeat',
    'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    'interpro_homologous_superfamily',
    'uniprot_functional_sites',
    'interpro_binding_site',
    'biomap_metal_ion_binding',
    'interpro_active_site',
    'uniprot_topology',
    'uniprot_post_translational_modification',
    'uniprot_phosphorylation',
    # 'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 7
# row_by = 'model_name'
# style_by = 
color_by = 'condition'

standard_cols = mean_df.columns[:8] + ['fold']

for model in model_order:


    # Combine standard columns
    score_dfs = []
    for score in scores_to_plot:
        # Select the intended score column, where there aren't nulls, and return it renamed as 'score
        score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
            pl.col(score).alias('score')
        ).select(standard_cols + ['score']))
    # Combine score dfs for different metrics
    score_df = pl.concat(score_dfs)

    # Filter for what we want to focus on

    all_filters = [
        pl.col('plm_embeddings_normalized') == False, # normalization filter
        pl.col('plm_state').is_in(plm_states), # plm training filter
        pl.col('linear_probe_state').is_in(probe_states), # probe training filter
        (pl.col('model_name') == model), # model filter
        pl.col('dataset').is_in(dataset_order), # dataset filter
        pl.col('control_type').is_in(show_controls), # control filter
        ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
    ]

    # Apply filters
    combined_filter = reduce(lambda a, b: a & b, all_filters)

    # Remap names for legibility
    df_filtered = score_df.filter(combined_filter).with_columns(
                pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
                pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
                pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
                pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian embedding', 'l2_normalized': 'normalized_embedding'}),
        ).with_columns(
                pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
        ).with_columns(
            pl.col("dataset").str.replace_all(dataset_regex, "") # Adjust dataset names for showing in figure
        ).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

    fig = sns.relplot(data = df_filtered, 
        x= xaxis_var, 
        y = 'score',
        col = col_by,
        col_wrap = col_wrap_num,
        # row = row_by,
        col_order = [re.sub(dataset_regex, "", i) for i in dataset_order],
        kind = 'line', 
        markers=True,
        errorbar='sd',
        # style=style_by,
        hue = color_by, 
        height = 2,
        palette = sns.color_palette("tab20", 15),
        facet_kws={'sharey': False},
        )
    fig.set_titles(col_template='{col_name}', row_template='{row_name}')

    residue_tasks = [
            'lipidation', 'topology', 'peptide', 'functional_sites', 
            'phosphorylation', 'secondary_structure', 'ssp_q3', 'ssp_q8', 
            'post_translational_modification'
        ]

    # 1. Strip regex from the category list to match the plot titles
    residue_tasks_cleaned = [re.sub(dataset_regex, "", t) for t in residue_tasks]

    # 2. Iterate through each subplot axis
    for ax in fig.axes.flat:
        title_text = ax.get_title().replace('_', ' ')
        
        # Check if this dataset is a residue-level task
        if title_text in residue_tasks_cleaned:
            # Style for Residue-level: Bold and Blue
            ax.set_title(title_text, 
            # fontweight='bold', 
            fontstyle='italic', 
            color='black', 
            fontsize=10)
        else:
            # Style for Protein-level: Bold and Dark Orange
            ax.set_title(title_text, 
            fontweight='bold', 
            color='black', 
            fontsize=10)

    fig.fig.suptitle(model_rename_dict.get(model))

    fig.set_axis_labels("layer #", "score")
    if fig._legend:
        leg = fig._legend
        leg.set_title('Condition')
        plt.setp(leg.get_title(), weight='bold')

    fig.tight_layout()
    plt.show()
    fig.savefig(linear_probe_figures_dir / f"probing_checkpoints_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}_controls.png", dpi=300)

### Plotting results for all checkpoint models

In [ ]:
# Harmonize 'score' column for plotting
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained', ] #'un-trained'
probe_states = ['trained', ] # 'un-trained'
show_controls = ['original'] # 'scrambled', 'mean', 'l2_normalized', 'random_gaussian'
# model_order = ['esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'amplify_120m',  'samplify_120m', 'amplify_350m', 'samplify_350m'] #'esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'amplify_120m',  'samplify_120m', 'amplify_350m', 'samplify_350m'
model_order = [
    # 'amplify_120m',
    'mila_amplify_120m_100000',
    'mila_amplify_120m_200000',
    'mila_amplify_120m_300000',
    'mila_amplify_120m_400000',
    'mila_amplify_120m_500000',
    'mila_amplify_120m_600000',
    'mila_amplify_120m_700000',
    'mila_amplify_120m_800000',
    'mila_amplify_120m_900000',
    'mila_amplify_120m_1000000'
    # 'mila_amplify_120m_u50_100000',
    # 'mila_amplify_120m_u50_200000',
    # 'mila_amplify_120m_u50_300000',
    # 'mila_amplify_120m_u50_400000',
    # 'mila_amplify_120m_u50_500000',
    # 'mila_amplify_120m_u50_600000',
    # 'mila_amplify_120m_u50_700000',
    # 'mila_amplify_120m_u50_800000',
    # 'mila_amplify_120m_u50_900000',
    # 'mila_amplify_120m_u50_1000000'
    ]

dataset_order = [
    'prot_param',
    # 'uniprot_peptide',
    'interpro_conserved_site',
    # 'biomap_localization_prediction',
    # 'interpro_repeat',
    # 'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    # 'interpro_homologous_superfamily',
    # 'uniprot_functional_sites',
    'interpro_binding_site',
    # 'biomap_metal_ion_binding',
    # 'interpro_active_site',
    # 'uniprot_topology',
    # 'uniprot_post_translational_modification',
    # 'uniprot_phosphorylation',
    # 'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

step_rename_dict = {
    'amplify_120m': 'AMPLIFY 120M',
    'samplify_120m': 'SaAMPLIFY 120M',
    'amplify_350m': 'AMPLIFY 350M',
    'samplify_350m': 'SaAMPLIFY 350M',
    'esm2_8m': 'ESM2 8M',
    'esm2_35m': 'ESM2 35M',
    'esm2_150m': 'ESM2 150M',
    'esm2_650m': 'ESM2 650M',
    'mila_amplify_120m_100000': '100k',
    'mila_amplify_120m_200000': '200k',
    'mila_amplify_120m_300000': '300k',
    'mila_amplify_120m_400000': '400k',
    'mila_amplify_120m_500000': '500k',
    'mila_amplify_120m_600000': '600k',
    'mila_amplify_120m_700000': '700k',
    'mila_amplify_120m_800000': '800k',
    'mila_amplify_120m_900000': '900k',
    'mila_amplify_120m_1000000': '1M',
    'mila_amplify_120m_u50_100000': 'AMPLIFY 120M UR50 100k',
    'mila_amplify_120m_u50_200000': 'AMPLIFY 120M UR50 200k',      
    'mila_amplify_120m_u50_300000': 'AMPLIFY 120M UR50 300k',
    'mila_amplify_120m_u50_400000': 'AMPLIFY 120M UR50 400k',
    'mila_amplify_120m_u50_500000': 'AMPLIFY 120M UR50 500k',
    'mila_amplify_120m_u50_600000': 'AMPLIFY 120M UR50 600k',
    'mila_amplify_120m_u50_700000': 'AMPLIFY 120M UR50 700k',
    'mila_amplify_120m_u50_800000': 'AMPLIFY 120M UR50 800k',
    'mila_amplify_120m_u50_900000': 'AMPLIFY 120M UR50 900k',
    'mila_amplify_120m_u50_1000000': 'AMPLIFY 120M UR50 1M'
}

step_colormapping = {
    'amplify_120m': '#FF0000',
    '100k': '#D3D3D3',
    '200k': '#BEBEBE',
    '300k': '#A9A9A9',
    '400k': '#949494',
    '500k': '#808080',
    '600k': '#6B6B6B',
    '700k': '#565656',
    '800k': '#414141',
    '900k': '#2C2C2C',
    '1M': '#171717',
}


xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 5
# row_by = 'model_name'
# style_by = 
color_by = 'model_name'

standard_cols = mean_df.columns[:8] + ['fold']

# Combine standard columns
score_dfs = []
for score in scores_to_plot:
    # Select the intended score column, where there aren't nulls, and return it renamed as 'score
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
# Combine score dfs for different metrics
score_df = pl.concat(score_dfs)

# Filter for what we want to focus on

all_filters = [
    pl.col('plm_embeddings_normalized') == False, # normalization filter
    pl.col('plm_state').is_in(plm_states), # plm training filter
    pl.col('linear_probe_state').is_in(probe_states), # probe training filter
    pl.col('model_name').is_in(model_order), # model filter
    pl.col('dataset').is_in(dataset_order), # dataset filter
    pl.col('control_type').is_in(show_controls), # control filter
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
]

# Apply filters
combined_filter = reduce(lambda a, b: a & b, all_filters)

# Remap names for legibility
df_filtered = score_df.filter(combined_filter).with_columns(
            pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
            pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
            pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
            pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
    ).with_columns(
            pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
    ).with_columns(
        pl.col("dataset").str.replace_all(dataset_regex, "") # Adjust dataset names for showing in figure
    ).sort(by=['plm_state', 'linear_probe_state', 'control_type']).sort(by='model_name', descending=False)

fig = sns.relplot(data = df_filtered.with_columns(pl.col('model_name').replace_strict(step_rename_dict)), 
    x= xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    # row = row_by,
    col_order = [re.sub(dataset_regex, "", i) for i in dataset_order],
    kind = 'line', 
    markers=True,
    errorbar='sd',
    # style=style_by,
    hue = color_by, 
    hue_order=[step_rename_dict.get(m) for m in model_order],
    palette = {k:v for k,v in step_colormapping.items()},
    alpha=0.8,
    height = 2,
    facet_kws={'sharey': False, 'sharex': False},
    )

fig.set_titles(col_template='{col_name}', row_template='{row_name}')


residue_tasks = [
        'lipidation', 'topology', 'peptide', 'functional_sites', 
        'phosphorylation', 'secondary_structure', 'ssp_q3', 'ssp_q8', 
        'post_translational_modification'
    ]

# 1. Strip regex from the category list to match the plot titles
residue_tasks_cleaned = [re.sub(dataset_regex, "", t) for t in residue_tasks]

# 2. Iterate through each subplot axis
for ax in fig.axes.flat:
    title_text = ax.get_title().replace('_', ' ')
    
    # Check if this dataset is a residue-level task
    if title_text in residue_tasks_cleaned:
        # Style for Residue-level: Bold and Blue
        ax.set_title(title_text, 
        # fontweight='bold', 
        fontstyle='italic', 
        color='black', 
        fontsize=10)
    else:
        # Style for Protein-level: Bold and Dark Orange
        ax.set_title(title_text, 
        fontweight='bold', 
        color='black', 
        fontsize=10)

fig.set_axis_labels("layer #", "score")
if fig._legend:
    leg = fig._legend
    leg.set_title('Training Steps')
    plt.setp(leg.get_title(), weight='bold')


fig.tight_layout()
fig.savefig(linear_probe_figures_dir / f"probing_checkpoints_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)

In [ ]:
# Harmonize 'score' column for plotting
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained', ] #'un-trained'
probe_states = ['trained', ] # 'un-trained'
show_controls = ['original'] # 'scrambled', 'mean', 'l2_normalized', 'random_gaussian'
# model_order = ['esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'amplify_120m',  'samplify_120m', 'amplify_350m', 'samplify_350m'] #'esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'amplify_120m',  'samplify_120m', 'amplify_350m', 'samplify_350m'
model_order = [
    # 'amplify_120m',
    # 'mila_amplify_120m_100000',
    # 'mila_amplify_120m_200000',
    # 'mila_amplify_120m_300000',
    # 'mila_amplify_120m_400000',
    # 'mila_amplify_120m_500000',
    # 'mila_amplify_120m_600000',
    # 'mila_amplify_120m_700000',
    # 'mila_amplify_120m_800000',
    # 'mila_amplify_120m_900000',
    'mila_amplify_120m_1000000',
    # 'mila_amplify_120m_u50_100000',
    # 'mila_amplify_120m_u50_200000',
    # 'mila_amplify_120m_u50_300000',
    # 'mila_amplify_120m_u50_400000',
    # 'mila_amplify_120m_u50_500000',
    # 'mila_amplify_120m_u50_600000',
    # 'mila_amplify_120m_u50_700000',
    # 'mila_amplify_120m_u50_800000',
    # 'mila_amplify_120m_u50_900000',
    'mila_amplify_120m_u50_1000000',
    ]

dataset_order = [
    'prot_param',
    # 'uniprot_peptide',
    'interpro_conserved_site',
    # 'biomap_localization_prediction',
    # 'interpro_repeat',
    # 'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    # 'interpro_homologous_superfamily',
    # 'uniprot_functional_sites',
    'interpro_binding_site',
    # 'biomap_metal_ion_binding',
    # 'interpro_active_site',
    # 'uniprot_topology',
    # 'uniprot_post_translational_modification',
    # 'uniprot_phosphorylation',
    # 'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

color_map = {
    'amplify_120m': '#FF0000', # Baseline Red
    
    # Standard Models: Purple/Violet (Starting darker)
    'mila_amplify_120m_100000':  '#C4B5FD', # Medium Light Purple
    'mila_amplify_120m_200000':  '#B19CFD',
    'mila_amplify_120m_300000':  '#A78BFA',
    'mila_amplify_120m_400000':  '#9061F9',
    'mila_amplify_120m_500000':  '#8B5CF6', # True Purple
    'mila_amplify_120m_600000':  '#7C3AED',
    'mila_amplify_120m_700000':  '#6D28D9',
    'mila_amplify_120m_800000':  '#5B21B6',
    'mila_amplify_120m_900000':  '#4C1D95',
    'mila_amplify_120m_1000000': '#3C0764', # Deep Indigo (Still Purple)

    # u50 Models: Green/Emerald (Starting darker)
    'mila_amplify_120m_u50_100000':  '#6EE7B7', # Medium Light Emerald
    'mila_amplify_120m_u50_200000':  '#34D399',
    'mila_amplify_120m_u50_300000':  '#10B981',
    'mila_amplify_120m_u50_400000':  '#059669',
    'mila_amplify_120m_u50_500000':  '#047857', # True Green
    'mila_amplify_120m_u50_600000':  '#065F46',
    'mila_amplify_120m_u50_700000':  '#064E3B',
    'mila_amplify_120m_u50_800000':  '#022C22',
    'mila_amplify_120m_u50_900000':  '#021F19',
    'mila_amplify_120m_u50_1000000': '#011511'  # Very Dark Green (Distinct from Deep Purple)
}

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 5
# row_by = 'model_name'
# style_by = 
color_by = 'model_name'

standard_cols = mean_df.columns[:8] + ['fold']

# Combine standard columns
score_dfs = []
for score in scores_to_plot:
    # Select the intended score column, where there aren't nulls, and return it renamed as 'score
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
# Combine score dfs for different metrics
score_df = pl.concat(score_dfs)

# Filter for what we want to focus on

all_filters = [
    pl.col('plm_embeddings_normalized') == False, # normalization filter
    pl.col('plm_state').is_in(plm_states), # plm training filter
    pl.col('linear_probe_state').is_in(probe_states), # probe training filter
    pl.col('model_name').is_in(model_order), # model filter
    pl.col('dataset').is_in(dataset_order), # dataset filter
    pl.col('control_type').is_in(show_controls), # control filter
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
]

# Apply filters
combined_filter = reduce(lambda a, b: a & b, all_filters)

# Remap names for legibility
df_filtered = score_df.filter(combined_filter).with_columns(
            pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
            pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
            pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
            pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
    ).with_columns(
            pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
    ).with_columns(
        pl.col("dataset").str.replace_all(dataset_regex, "") # Adjust dataset names for showing in figure
    ).sort(by=['plm_state', 'linear_probe_state', 'control_type']).sort(by='model_name', descending=False)

fig = sns.relplot(data = df_filtered.with_columns(pl.col('model_name').replace_strict(model_rename_dict)), 
    x= xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    # row = row_by,
    col_order = [re.sub(dataset_regex, "", i) for i in dataset_order],
    kind = 'line', 
    markers=True,
    errorbar='sd',
    # style=style_by,
    hue = color_by, 
    hue_order=[model_rename_dict.get(m) for m in model_order],
    palette = {model_rename_dict.get(k):v for k,v in color_map.items()},
    alpha=0.8,
    height = 2,
    facet_kws={'sharey': False, 'sharex': False},
    )

fig.set_titles(col_template='{col_name}', row_template='{row_name}')

residue_tasks = [
        'lipidation', 'topology', 'peptide', 'functional_sites', 
        'phosphorylation', 'secondary_structure', 'ssp_q3', 'ssp_q8', 
        'post_translational_modification'
    ]

# 1. Strip regex from the category list to match the plot titles
residue_tasks_cleaned = [re.sub(dataset_regex, "", t) for t in residue_tasks]

# 2. Iterate through each subplot axis
for ax in fig.axes.flat:
    title_text = ax.get_title().replace('_', ' ')
    
    # Check if this dataset is a residue-level task
    if title_text in residue_tasks_cleaned:
        # Style for Residue-level: Bold and Blue
        ax.set_title(title_text, 
        # fontweight='bold', 
        fontstyle='italic', 
        color='black', 
        fontsize=10)
    else:
        # Style for Protein-level: Bold and Dark Orange
        ax.set_title(title_text, 
        fontweight='bold', 
        color='black', 
        fontsize=10)

fig.set_axis_labels("layer #", "score")
if fig._legend:
    leg = fig._legend
    leg.set_title('Model')
    plt.setp(leg.get_title(), weight='bold')

fig.tight_layout()
fig.savefig(linear_probe_figures_dir / f"probing_checkpoints_u50_100_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)